In [1]:
import pandas as pd
import numpy as np

# Load data
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"
df = pd.read_csv(url)

# Make dataset bigger
df_big = pd.concat([df]*50, ignore_index=True)

# Convert to retail dataset
df_big["date"] = pd.to_datetime(df_big["year"].astype(str) + "-" + df_big["month"] + "-01")
df_big["store_id"] = np.random.randint(1, 6, size=len(df_big))
df_big["product_id"] = np.random.randint(100, 200, size=len(df_big))

df_big["demand"] = df_big["passengers"]
df_big = df_big.drop(columns=["passengers"])

df_big.head()

,year,month,date,store_id,product_id,demand
0,1949,January,1949-01-01,2,134,112
1,1949,February,1949-02-01,3,176,118
2,1949,March,1949-03-01,4,161,132
3,1949,April,1949-04-01,1,112,129
4,1949,May,1949-05-01,1,173,121


In [2]:
import pandas as pd
import numpy as np

# Load base data
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"
df = pd.read_csv(url)

# Make dataset slightly bigger
df_big = pd.concat([df]*20, ignore_index=True)

# Convert to retail format
df_big["date"] = pd.to_datetime(df_big["year"].astype(str) + "-" + df_big["month"] + "-01")

df_big["store_id"] = np.random.randint(1, 6, size=len(df_big))
df_big["product_id"] = np.random.randint(100, 200, size=len(df_big))

df_big["demand"] = df_big["passengers"]

# Remove old column
df_big = df_big.drop(columns=["passengers"])

# Show result
df_big.head()

,year,month,date,store_id,product_id,demand
0,1949,January,1949-01-01,2,121,112
1,1949,February,1949-02-01,5,149,118
2,1949,March,1949-03-01,5,123,132
3,1949,April,1949-04-01,3,127,129
4,1949,May,1949-05-01,1,189,121


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Save dataset into your project folder
df_big.to_csv("/content/drive/MyDrive/retail-demand-intelligence/retail_clean.csv", index=False)

print("✅ Dataset saved")

MessageError: Error: credential propagation was unsuccessful

In [4]:
# Add price (random between 10 and 100)
df_big["price"] = np.random.uniform(10, 100, size=len(df_big))

# Calculate revenue
df_big["revenue"] = df_big["demand"] * df_big["price"]

df_big.head()


,year,month,date,store_id,product_id,demand,price,revenue
0,1949,January,1949-01-01,2,121,112,46.690328,5229.316694
1,1949,February,1949-02-01,5,149,118,86.559454,10214.015561
2,1949,March,1949-03-01,5,123,132,35.996852,4751.584429
3,1949,April,1949-04-01,3,127,129,31.756756,4096.621570
4,1949,May,1949-05-01,1,189,121,23.038139,2787.614810


In [5]:
# Sort dataset (important for time series)
df_big = df_big.sort_values(["store_id", "product_id", "date"])

# Rolling average (trend)
df_big["rolling_3"] = df_big.groupby(["store_id","product_id"])["demand"].transform(lambda x: x.rolling(3).mean())

# Lag features (past demand)
df_big["lag_1"] = df_big.groupby(["store_id","product_id"])["demand"].shift(1)
df_big["lag_2"] = df_big.groupby(["store_id","product_id"])["demand"].shift(2)

# Growth rate (change)
df_big["growth_rate"] = df_big.groupby(["store_id","product_id"])["demand"].pct_change()

df_big.head()


,year,month,date,store_id,product_id,demand,price,revenue,rolling_3,lag_1,lag_2,growth_rate
487,1953,August,1953-08-01,1,100,272,18.631840,5067.860355,NaN,NaN,NaN,NaN
505,1955,February,1955-02-01,1,100,233,48.367695,11269.673034,NaN,272.0,NaN,-0.143382
257,1958,June,1958-06-01,1,100,435,15.655058,6809.950210,313.333333,233.0,272.0,0.866953
1903,1951,August,1951-08-01,1,101,199,19.771092,3934.447243,NaN,NaN,NaN,NaN
493,1954,February,1954-02-01,1,101,188,54.532282,10252.069048,NaN,199.0,NaN,-0.055276


In [6]:
# Sort data (IMPORTANT)
df_big = df_big.sort_values(["store_id", "product_id", "date"])

# Rolling trend
df_big["rolling_3"] = df_big.groupby(["store_id", "product_id"])["demand"].transform(lambda x: x.rolling(3).mean())

# Lag features
df_big["lag_1"] = df_big.groupby(["store_id", "product_id"])["demand"].shift(1)
df_big["lag_2"] = df_big.groupby(["store_id", "product_id"])["demand"].shift(2)

# Growth rate
df_big["growth_rate"] = df_big.groupby(["store_id", "product_id"])["demand"].pct_change()

df_big.head()


,year,month,date,store_id,product_id,demand,price,revenue,rolling_3,lag_1,lag_2,growth_rate
487,1953,August,1953-08-01,1,100,272,18.631840,5067.860355,NaN,NaN,NaN,NaN
505,1955,February,1955-02-01,1,100,233,48.367695,11269.673034,NaN,272.0,NaN,-0.143382
257,1958,June,1958-06-01,1,100,435,15.655058,6809.950210,313.333333,233.0,272.0,0.866953
1903,1951,August,1951-08-01,1,101,199,19.771092,3934.447243,NaN,NaN,NaN,NaN
493,1954,February,1954-02-01,1,101,188,54.532282,10252.069048,NaN,199.0,NaN,-0.055276


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Remove missing values (important)
df_model = df_big.dropna()

# Select features (inputs)
X = df_model[["lag_1", "lag_2", "rolling_3", "growth_rate"]]

# Target (what we predict)
y = df_model["demand"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Train model
model = RandomForestRegressor()
model.fit(X_train, y_train)

print("✅ Model trained")


✅ Model trained


In [8]:
from sklearn.metrics import mean_absolute_error

# Predict
y_pred = model.predict(X_test)

# Calculate error
mae = mean_absolute_error(y_test, y_pred)

print("✅ MAE:", mae)


✅ MAE: 6.952842105263158


In [9]:
# Recommendation logic
def get_recommendation(g):
    if g > 0.05:
        return "Increase stock"
    elif g < 0.01:
        return "Reduce inventory"
    else:
        return "Monitor"

# Apply to dataset
df_big["recommendation"] = df_big["growth_rate"].apply(get_recommendation)

df_big.head()

,year,month,date,store_id,product_id,demand,price,revenue,rolling_3,lag_1,lag_2,growth_rate,recommendation
487,1953,August,1953-08-01,1,100,272,18.631840,5067.860355,NaN,NaN,NaN,NaN,Monitor
505,1955,February,1955-02-01,1,100,233,48.367695,11269.673034,NaN,272.0,NaN,-0.143382,Reduce inventory
257,1958,June,1958-06-01,1,100,435,15.655058,6809.950210,313.333333,233.0,272.0,0.866953,Increase stock
1903,1951,August,1951-08-01,1,101,199,19.771092,3934.447243,NaN,NaN,NaN,NaN,Monitor
493,1954,February,1954-02-01,1,101,188,54.532282,10252.069048,NaN,199.0,NaN,-0.055276,Reduce inventory


In [10]:
df_big["recommendation"].value_counts()

,count
recommendation,
Increase stock,1672
Monitor,614
Reduce inventory,594


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_big.to_csv(
    "/content/drive/MyDrive/retail-demand-intelligence/retail_clean.csv",
    index=False
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
